In [ ]:
# Make this notebook runnable from any working directory: locate the repository
# root by its marker file, then work from this notebook's own folder, which is
# what the relative paths below assume.
import os
from pathlib import Path
_root = Path.cwd().resolve()
while not (_root / "requirements.txt").is_file() and _root != _root.parent:
    _root = _root.parent
_here = _root / "refinement-per-claim" / "human_validation-revised"
if not _here.is_dir():
    raise RuntimeError(
        "Could not locate " + str(_here) + ". Run this notebook from inside the "
        "cloned repository."
    )
os.chdir(_here)

# Human Validation — Build Comparison Pairs

Generates 480 comparison pairs (20 claims × 3 KPIs × 8 pairs each) using:
- **Generator**: Claude Haiku 3.5 via direct Anthropic API
- **Initial prompts**: generic prompts from `pro_ukrainian_agents.py`
- **Refined prompts**: claim-specific prompts from `refined_system_prompts.xlsx`, the refined prompts with the naturalness and coherence guidelines added after the initial validation study

Pair structure per claim × KPI:
- 4 high-diff pairs: refined vs. initial (left_better = 1 if refined is left, else 0)
- 2 low-diff pairs: initial vs. initial (left_better = None)
- 2 low-diff pairs: refined vs. refined (left_better = None)

## Imports

In [ ]:
import os
import sys
import random
import time
import numpy as np
import pandas as pd

sys.path.insert(0, os.path.abspath(".."))
from pro_ukrainian_agents import (
    create_pro_ukrainian_agents_claude,
    pro_ukrainian_agents_system_prompts,
    pro_ukrainian_agents_descriptions
)

DATA_DIR          = r"../../Data"
REFINED_PROMPTS_PATH = r"refined_system_prompts.xlsx"
NARRATIVES_PATH   = os.path.join(DATA_DIR, "Pro Russian top users and narratives.xlsx")
NARRATIVES_SHEET  = "top_20_narratives_20250313_1716"
OUTPUT_PATH       = r"mini_exp_revised_comparisons.xlsx"

MODEL_NAME        = "openrouter/anthropic/claude-3.5-haiku"
TEMP              = 0.5
N_CNS_PER_COND    = 4   # CNs generated per condition per claim x KPI
RANDOM_SEED       = 42

## Load Claims & Refined Prompts

In [ ]:
narratives = pd.read_excel(
    NARRATIVES_PATH, sheet_name=NARRATIVES_SHEET
)["description"].tolist()
print(f"Claims loaded: {len(narratives)}")

refined_prompts_df = pd.read_excel(REFINED_PROMPTS_PATH)
print(f"Refined prompts loaded: {len(refined_prompts_df)} rows")
refined_prompts_df.head()

## Create Initial CN Agents

These are claim-agnostic — same prompts for all claims.

In [ ]:
initial_agents = create_pro_ukrainian_agents_claude(
    descriptions=pro_ukrainian_agents_descriptions,
    system_prompts=pro_ukrainian_agents_system_prompts,
    model_name=MODEL_NAME,
    temp=TEMP
)
print(f"Initial agents: {list(initial_agents.keys())}")

Initial agents: ['CN_Creator_Agent_1', 'CN_Creator_Agent_2', 'CN_Creator_Agent_3', 'CN_Creator_Agent_4']


## CN Generation Helper

In [ ]:
def generate_diverse_cns(claim: str, agent, n: int = 4, retries: int = 3) -> list[str]:
    """
    Generates n diverse CNs for a claim using the given agent.
    Each subsequent call includes previous responses to nudge diversity.
    Returns fewer than n items if some calls fail.
    """
    cns = []
    for _ in range(n):
        if not cns:
            prompt = claim
        else:
            prev = "\n".join([f"{j+1}. {c}" for j, c in enumerate(cns)])
            prompt = (
                claim
                + "\n\nHere are your previous responses:\n"
                + prev
                + "\n\nThink carefully and generate a response different from your previous ones "
                "(in terms of structure, content, etc.) while strictly following your KEY GUIDELINES."
            )
        for attempt in range(retries):
            try:
                response = agent.run(prompt, reset=True)
                cns.append(response)
                break
            except Exception as e:
                print(f"    Generation error (attempt {attempt+1}): {e}")
                time.sleep(2)
    return cns

## Pair Building Helper

Builds 8 pairs from 4 initial + 4 refined CNs, with constraints:
- Each CN used at most 2 times across all pairs
- No duplicate (unordered) pairs
- Left/right side randomized

In [11]:
def build_pairs(claim, initial_cns, refined_cns, kpi_id, rng, max_retries=2000):
    """
    Builds 8 pairs:
      4 high-diff  (R vs I)
      2 low-diff   (I vs I)
      2 low-diff   (R vs R)
    Returns list of row dicts, or raises RuntimeError if constraints can't be met.
    """
    pairs_spec = (
        [("high", "R", "I")] * 4 +
        [("low",  "I", "I")] * 2 +
        [("low",  "R", "R")] * 2
    )

    I_keys = [("I", idx) for idx in range(len(initial_cns))]
    R_keys = [("R", idx) for idx in range(len(refined_cns))]
    cn_text = lambda k: refined_cns[k[1]] if k[0] == "R" else initial_cns[k[1]]

    for _ in range(max_retries):
        usage = {k: 0 for k in I_keys + R_keys}
        rng.shuffle(pairs_spec)
        tmp_rows = []
        seen_pairs = set()
        ok = True

        for diff_level, g1, g2 in pairs_spec:
            found = False
            for _try in range(100):
                pool1 = [k for k in (R_keys if g1 == "R" else I_keys) if usage[k] < 2]
                pool2 = [k for k in (R_keys if g2 == "R" else I_keys) if usage[k] < 2]
                if not pool1 or not pool2:
                    break

                k1 = rng.choice(pool1)
                k2 = rng.choice(pool2)

                if g1 == g2 and k1 == k2:
                    continue

                pair_id = tuple(sorted([k1, k2]))
                if pair_id in seen_pairs:
                    continue

                usage[k1] += 1
                usage[k2] += 1
                seen_pairs.add(pair_id)

                if rng.random() < 0.5:
                    left, right = k1, k2
                else:
                    left, right = k2, k1

                left_better = None
                if diff_level == "high":
                    left_better = 1 if left[0] == "R" else 0

                tmp_rows.append({
                    "claim":      claim,
                    "cn_1":       cn_text(left),
                    "cn_2":       cn_text(right),
                    "diff_level": diff_level,
                    "left_better": left_better,
                    "kpi_id":     kpi_id
                })
                found = True
                break

            if not found:
                ok = False
                break

        if ok:
            return tmp_rows

    raise RuntimeError(
        f"Could not satisfy pair constraints for claim='{claim[:50]}', kpi_id={kpi_id}. "
        "Consider generating more CNs or relaxing constraints."
    )

## Main Loop — Generate CNs & Build Pairs

For each of 20 claims × 3 KPIs:
1. Generate 4 initial CNs (generic prompt)
2. Build a claim-specific refined agent from the refined prompts
3. Generate 4 refined CNs
4. Build 8 comparison pairs

In [ ]:
KPI_MAP = {
    1: "Persuasiveness",
    2: "Emotional Engagement",
    3: "Shareability"
}
AGENT_KEY = {
    1: "CN_Creator_Agent_1",
    2: "CN_Creator_Agent_2",
    3: "CN_Creator_Agent_3"
}

rng = random.Random(RANDOM_SEED)

# Load existing results to support resuming after a failure
if os.path.exists(OUTPUT_PATH):
    df_existing = pd.read_excel(OUTPUT_PATH)
    all_rows = df_existing.to_dict("records")
    done = set(zip(df_existing["claim"], df_existing["kpi_id"].astype(int)))
    print(f"Resuming: {len(df_existing)} rows already saved, {len(done)//8} claim×KPI combos done.")
else:
    all_rows = []
    done = set()

skipped = []

for claim_idx, narrative in enumerate(narratives):
    claim_number = claim_idx + 1
    print(f"\n[Claim {claim_number}/{len(narratives)}] {narrative[:80]}...")

    for kpi_id in [1, 2, 3]:
        if (narrative, kpi_id) in done:
            print(f"  [{KPI_MAP[kpi_id]}] Already done — skipping.")
            continue

        kpi_name  = KPI_MAP[kpi_id]
        agent_key = AGENT_KEY[kpi_id]
        print(f"  [{kpi_name}]")

        try:
            # --- Initial CNs ---
            print(f"    Generating {N_CNS_PER_COND} initial CNs...")
            initial_cns = generate_diverse_cns(narrative, initial_agents[agent_key], N_CNS_PER_COND)
            print(f"    Got {len(initial_cns)} initial CNs.")

            # --- Refined agent (claim-specific) ---
            refined_prompt_rows = refined_prompts_df[
                (refined_prompts_df["ClaimNumber"] == claim_number) &
                (refined_prompts_df["KPI"] == kpi_name)
            ]["SystemPrompt"].values

            if len(refined_prompt_rows) == 0:
                print(f"    WARNING: No refined prompt found — skipping this KPI.")
                continue

            refined_agent_dict = create_pro_ukrainian_agents_claude(
                descriptions={agent_key: pro_ukrainian_agents_descriptions[agent_key]},
                system_prompts={agent_key: refined_prompt_rows[0]},
                model_name=MODEL_NAME,
                temp=TEMP
            )
            refined_agent = refined_agent_dict[agent_key]

            # --- Refined CNs ---
            print(f"    Generating {N_CNS_PER_COND} refined CNs...")
            refined_cns = generate_diverse_cns(narrative, refined_agent, N_CNS_PER_COND)
            print(f"    Got {len(refined_cns)} refined CNs.")

            # --- Build pairs ---
            pairs = build_pairs(narrative, initial_cns, refined_cns, kpi_id, rng)
            all_rows.extend(pairs)
            done.add((narrative, kpi_id))
            print(f"    Built {len(pairs)} pairs.")

            # Save after every claim×KPI so progress is never lost
            pd.DataFrame(all_rows).to_excel(OUTPUT_PATH, index=False)

        except RuntimeError as e:
            skipped.append((claim_number, kpi_name))

df = pd.DataFrame(all_rows)
print(f"\nDone. Total pairs: {len(df)} — saved to {OUTPUT_PATH}")
if skipped:
    print(f"\nSkipped: {skipped}")
df.groupby(["kpi_id", "diff_level"]).size()

## Sort & Save

Sort by KPI then by claim (stable), so evaluators see all pairs for one KPI together.

In [ ]:
df = pd.read_excel(OUTPUT_PATH)

df_sorted = df.sort_values(
    by=["kpi_id", "claim"],
    kind="mergesort"
).reset_index(drop=True)

df_sorted["left_better"] = df_sorted["left_better"].fillna(0).astype(int)

df_sorted.to_excel(OUTPUT_PATH, index=False)

print(f"Sorted and saved: {len(df_sorted)} rows")
print(f"\nHigh-diff balance (left_better mean, should be ~0.5):")
print(df_sorted[df_sorted["diff_level"] == "high"].groupby("kpi_id")["left_better"].mean())
df_sorted.head(10)